<a href="https://colab.research.google.com/github/hemidovaqil/turboaz-car-price-prediction/blob/main/Preprocessing%20Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2. Data Preprocessing Pipeline

At this stage, the data is split into training and test sets prior to preprocessing. For numerical features, missing values ​​are imputed using the median, followed by standardization with `StandardScaler`. For categorical features, missing values ​​are imputed using the most frequent value, followed by One-Hot Encoding. All steps are combined using `Pipeline` and `ColumnTransformer`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('/content/cars.csv', engine='python', on_bad_lines='skip')
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Dataset shape: (88284, 56)
Columns: ['id_x', 'car_rel_url_x', 'datetime_scrape', 'name', 'price_x', 'currency_x', 'datetime_product', 'city', 'day', 'hour', 'attributes', 'production_year', 'engine_displacement_num', 'engine_displacement_unit', 'kilometrage_num', 'kilometrage_unit', 'barter', 'loan', 'salon', 'spare_parts', 'vip', 'featured', 'img_url', 'id_y', 'cars_id', 'car_rel_url_y', 'datetime', 'description', 'price_y', 'currency_y', 'owner_name', 'shop_name', 'phone', 'updated', 'views', 'vin', 'car_details_id_x', 'Ban növü', 'Buraxılış ili', 'Hansı bazar üçün yığılıb', 'Marka', 'Model', 'Mühərrik', 'Qəzalı', 'Rəng', 'Sahiblər', 'Sürətlər qutusu', 'Vəziyyəti', 'Yeni', 'Yerlərin sayı', 'Yürüş', 'Ötürücü', 'Şəhər', 'car_details_id_y', 'car_rel_url', 'extra_info']


,id_x,car_rel_url_x,datetime_scrape,name,price_x,currency_x,datetime_product,city,day,hour,...,Sürətlər qutusu,Vəziyyəti,Yeni,Yerlərin sayı,Yürüş,Ötürücü,Şəhər,car_details_id_y,car_rel_url,extra_info
0,3c234145-d57a-4ad6-9448-d43810fc3392,/autos/8748840-hyundai-i30,2024-09-13 20:32:19.751157+00,Hyundai i30,15000.0,AZN,"Bakı, dünən 23:28",bakı,13.09.2024,23:28,...,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Xeyr,5,270 000 km,Ön,Bakı,8d84d800-fafd-4d5c-b640-f47bd6c5ac20,/autos/8748840-hyundai-i30,Yüngül lehimli disklər* ABS* Mərkəzi qapanma* ...
1,c74ea36f-6be1-4de4-926d-e117197dcf00,/autos/8475807-lada-vaz-niva-travel,2024-09-13 20:32:19.751157+00,LADA (VAZ) Niva Travel,23700.0,AZN,"Bakı, dünən 23:59",bakı,13.09.2024,23:59,...,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Bəli,5,0 km,Tam,Bakı,2cf8b84b-adaf-467a-8f06-3dabcf866c8a,/autos/8475807-lada-vaz-niva-travel,Yüngül lehimli disklər* ABS* Kondisioner* Otur...
2,9cefceb0-024d-4581-a869-a3c2c68a9f95,/autos/8739686-toyota-land-cruiser,2024-09-13 20:32:19.751157+00,Toyota Land Cruiser,35600.0,$,"Bakı, dünən 23:59",bakı,13.09.2024,23:59,...,Avtomat,"Vuruğu yoxdur, rənglənməyib",Xeyr,8+,164 750 km,Tam,Bakı,22bb3704-ebe7-4357-ba2f-1806d1a7042b,/autos/8739686-toyota-land-cruiser,Yüngül lehimli disklər* ABS* Lyuk* Mərkəzi qap...
3,459cc337-fb63-48de-9694-41554923d311,/autos/8712597-hyundai-elantra,2024-09-13 20:32:19.751157+00,Hyundai Elantra,26700.0,AZN,"Bakı, dünən 23:59",bakı,13.09.2024,23:59,...,Avtomat,"Vuruğu yoxdur, rənglənməyib",Xeyr,NaN,126 000 km,Ön,Bakı,e0d16dac-4091-4417-916e-cadeff600f95,/autos/8712597-hyundai-elantra,Yüngül lehimli disklər* ABS* Lyuk* Yağış senso...
4,6c5ee8d8-1c6f-4fad-a694-957a4c43c25d,/autos/8674773-toyota-prius,2024-09-13 20:32:19.751157+00,Toyota Prius,10500.0,AZN,"Bakı, dünən 23:59",bakı,13.09.2024,23:59,...,Variator,"Vuruğu yoxdur, rənglənməyib",Xeyr,5,354 000 km,Ön,Bakı,82078856-0d07-4a0d-8123-69c2e5002e07,/autos/8674773-toyota-prius,Arxa görüntü kamerası


In [ ]:
# Rename columns for better readability and consistency with English translation
column_name_mapping = {
    "Marka": "Make",
    "Model": "Model",
    "Rəng": "Color",
    "Sürətlər qutusu": "Gearbox",
    "Ötürücü": "Drive_Type",
    "Yeni": "New",
    "Sahiblər": "Owners",
    "Qəzalı": "Damaged",
    "Buraxılış ili": "Production_Year",
    "Yürüş": "Mileage",
    "Yerlərin sayı": "Number_of_Seats",
    "currency_x": "Currency"
}

df.rename(columns=column_name_mapping, inplace=True)

print("Columns renamed to English equivalents.")
print("New DataFrame columns (first 20):", df.columns.tolist()[:20])

Columns renamed to English equivalents.
New DataFrame columns (first 20): ['id_x', 'car_rel_url_x', 'datetime_scrape', 'name', 'price_x', 'Currency', 'datetime_product', 'city', 'day', 'hour', 'attributes', 'production_year', 'engine_displacement_num', 'engine_displacement_unit', 'kilometrage_num', 'kilometrage_unit', 'barter', 'loan', 'salon', 'spare_parts']


In [ ]:
import re

# Clean 'Mileage' column
# Remove ' km' and spaces, then convert to numeric
df['Mileage'] = df['Mileage'].astype(str).apply(lambda x: re.sub(r'\s*km', '', x)).str.replace(' ', '')
df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')

print("Cleaned 'Mileage' column:")
display(df['Mileage'].head())

Cleaned 'Mileage' column:


,Mileage
0,270000
1,0
3,126000
4,354000
6,275642


In [ ]:
TARGET = "price_x"

# Check for the new 'Currency' column
if "Currency" in df.columns:
    print(df["Currency"].value_counts())
    # Filter DataFrame to include only cars with AZN currency
    df = df[df["Currency"] == "AZN"]

# Define categorical and numerical columns using the new English names
CATEGORICAL_COLS = [
    "Make", "Model", "Color", "Gearbox",
    "Drive_Type", "New", "Owners", "Damaged",
]
NUMERIC_COLS = [
    "Production_Year", "Mileage", "Number_of_Seats",
]

# Filter columns to ensure they exist in the DataFrame
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in df.columns]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df.columns]
FEATURES = CATEGORICAL_COLS + NUMERIC_COLS

# Drop rows where the target variable is missing
df = df.dropna(subset=[TARGET])

# Redefine features (X) and target (y) based on the filtered DataFrame and selected features
X = df[FEATURES]
y = df[TARGET]

print("Shape of X after filtering and feature selection:", X.shape)
print("Shape of y after filtering:", y.shape)

Currency
AZN    62866
Name: count, dtype: int64
Shape of X after filtering and feature selection: (62866, 11)
Shape of y after filtering: (62866,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (50292, 11)  Test: (12574, 11)


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, NUMERIC_COLS),
    ("cat", categorical_pipeline, CATEGORICAL_COLS),
])

### Conclusion

The data was split into training and test sets prior to preprocessing. Numerical features were standardized using StandardScaler, while categorical features were transformed via One-Hot Encoding. Missing values ​​were handled within the pipeline. Using ColumnTransformer allows for the application of different processing methods to different feature types and reduces the risk of data leakage.